In [1]:
from KNN import KNN
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os



In [2]:
#define helper functions
#function to evaluate model's accuracy
def evaluate_acc(y_true, y_pred):
    total_samples = len(y_true)
    correct_predictions = np.sum(y_true == y_pred)
    return (correct_predictions / total_samples) 

#function to create train test split
def create_train_test_split(x, y, test_ratio):
    n_samples = x.shape[0]

    # shuffle indices
    indices = np.random.permutation(n_samples)

    # compute split point
    test_count = int(n_samples * test_ratio)

    test_idx = indices[:test_count] #get indices for test set
    train_idx = indices[test_count:] #get indices for train set

    #create train and test sets
    x_train = x.iloc[train_idx]
    x_test = x.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]


    return x_train, x_test, y_train, y_test

#Dimensionality reduction function using PCA (Principal Component Analysis)
def get_pca_projection(X, n_components=2):
    #standardize
    X_std = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

    #covariance matrix
    cov_mat = np.cov(X_std.T)

    #eigenvalues and eigenvectors
    eig_vals, eig_vecs = np.linalg.eig(cov_mat)

    #get top eigenvectors
    idx = np.argsort(eig_vals)[::-1]
    top_vectors = eig_vecs[:, idx[:n_components]]

    # Return the projected data and the vectors 
    return X_std.dot(top_vectors), top_vectors

#Plotting function to visualize decision boundary of kNN model
def plot_decision_boundary(model, X, y, title):
    #get 2Dfeatures
    X_arr = np.array(X)
    X_2d, _ = get_pca_projection(X_arr, n_components=2)
    y_arr = np.array(y)

    #create mesh grid
    h = 1.0  # Keep this large for faster plotting
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), 
                         np.arange(y_min, y_max, h))

    # Train a new kNN model on the 2D data for fast prediction
    model_2d = KNN()
    model_2d.fit(X_2d, y, model.k_neighbours, model.distance)
    
    #Predictions on the grid points
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = np.array(model_2d.predict(grid_points))
    Z = Z.reshape(xx.shape)

    #Plotting
    plt.figure(figsize=(8, 5))
    plt.contourf(xx, yy, Z,levels=[-0.5, 0.5, 1.5], alpha=0.3, cmap='coolwarm')
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, edgecolor='k', s=20, cmap='coolwarm')
    plt.title(title)

    # 6. Save the file
    folder = "kNN_plots"
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    filename = title.replace(" ", "_").replace(":", "").replace(",", "")
    save_path = f"{folder}/{filename}.png"
    
    plt.savefig(save_path)
    plt.close() # Free up memory
    print(f"Plot saved to: {save_path}")


In [3]:
#load datasets
breast_cancer_df = pd.read_csv(r"data_cleaned\breast_cancer.csv")
hepatitis_df = pd.read_csv(r"data_cleaned\hepatitis.csv")
hepatitis_imputated_df = pd.read_csv(r"data_cleaned\hepatitis_imputed.csv")

#create training and testing sets
#breast cancer dataset
x_bc = breast_cancer_df.drop(columns="diagnosis")
y_bc = breast_cancer_df["diagnosis"]
x_train_bc, x_test_bc, y_train_bc, y_test_bc = create_train_test_split(x_bc, y_bc, 0.2)

#hepatitis dataset
x_hp = hepatitis_df.drop(columns="class")
y_hp = hepatitis_df["class"]
x_train_hp, x_test_hp, y_train_hp, y_test_hp = create_train_test_split(x_hp, y_hp, 0.2)

#hepatitis dataset with imputation
x_hp_imp = hepatitis_imputated_df.drop(columns="class")
y_hp_imp = hepatitis_imputated_df["class"]
x_train_hp_imp, x_test_hp_imp, y_train_hp_imp, y_test_hp_imp = create_train_test_split(x_hp_imp, y_hp_imp, 0.2)




In [4]:
#Train the models
k_values = [1, 3, 5, 7, 9, 11, 13, 15] #different k values to test
distances = ["euclidean", "manhattan", "minkowski"] #different distance metrics to test

for dis in distances:
    print(f"\n---------------Evaluating models using {dis} distance---------------")
    for k in k_values:
            
        bc_model = KNN() #define models
        hp_model = KNN() #define models
        hp_model_imp = KNN() #define models

        #train the models
        bc_model.fit(x_train_bc,y_train_bc, k, dis)
        hp_model.fit(x_train_hp, y_train_hp, k, dis)
        hp_model_imp.fit(x_train_hp_imp, y_train_hp_imp, k, dis)

        #TO DO plot decision boundary for each model
        plot_decision_boundary(bc_model, x_train_bc, y_train_bc, f"BC Model: k={k}, dist={dis}")
        plot_decision_boundary(hp_model, x_train_hp, y_train_hp, f"HP Model: k={k}, dist={dis}")
        plot_decision_boundary(hp_model_imp, x_train_hp_imp, y_train_hp_imp, f"HP Imp Model: k={k}, dist={dis}")

        #Test
        y_pred_bc = bc_model.predict(x_test_bc)
        y_pred_hp = hp_model.predict(x_test_hp)
        y_pred_hp_imp = hp_model_imp.predict(x_test_hp_imp)

        #Save predictions
        df_bc = pd.DataFrame({
            "y_test": y_test_bc,
            "y_pred": y_pred_bc
        })
        df_bc.to_csv(f"predictions_knn/bc_predictions_k{k}_{dis}.csv", index=False)

        df_hp = pd.DataFrame({
            "y_test": y_test_hp,
            "y_pred": y_pred_hp
        })
        df_hp.to_csv(f"predictions_knn/hp_predictions_k{k}_{dis}.csv", index=False)

        df_hp_imp = pd.DataFrame({
            "y_test": y_test_hp_imp,
            "y_pred": y_pred_hp_imp
        })
        df_hp_imp.to_csv(f"predictions_knn/hp_imp_predictions_k{k}_{dis}.csv", index=False)
        

        #Evaluate
        print(f"Accuracy of bc_model using {k} neighbours and {dis} distance: {evaluate_acc(y_test_bc, y_pred_bc)}")
        print(f"Accuracy of hp_model using {k} neighbours and {dis} distance: {evaluate_acc(y_test_hp, y_pred_hp)}")
        print(f"Accuracy of hp_model_imp using {k} neighbours and {dis} distance: {evaluate_acc(y_test_hp_imp, y_pred_hp_imp)}\n")



---------------Evaluating models using euclidean distance---------------
Plot saved to: kNN_plots/BC_Model_k=1_dist=euclidean.png
Plot saved to: kNN_plots/HP_Model_k=1_dist=euclidean.png
Plot saved to: kNN_plots/HP_Imp_Model_k=1_dist=euclidean.png
Accuracy of bc_model using 1 neighbours and euclidean distance: 0.9026548672566371
Accuracy of hp_model using 1 neighbours and euclidean distance: 0.75
Accuracy of hp_model_imp using 1 neighbours and euclidean distance: 0.7096774193548387

Plot saved to: kNN_plots/BC_Model_k=3_dist=euclidean.png
Plot saved to: kNN_plots/HP_Model_k=3_dist=euclidean.png
Plot saved to: kNN_plots/HP_Imp_Model_k=3_dist=euclidean.png
Accuracy of bc_model using 3 neighbours and euclidean distance: 0.911504424778761
Accuracy of hp_model using 3 neighbours and euclidean distance: 0.75
Accuracy of hp_model_imp using 3 neighbours and euclidean distance: 0.6451612903225806

Plot saved to: kNN_plots/BC_Model_k=5_dist=euclidean.png
Plot saved to: kNN_plots/HP_Model_k=5_di